# Analysis of the distance


In [1]:
import os
import sys
from base_fns import get_local_file
file = get_local_file()

local_dir = os.path.dirname(os.path.dirname(file))
sys.path.append(local_dir)

import json
import dill
import numpy as np
import matplotlib.pyplot as plt
import numpy as np

from config import Config
from tools.distance import DistanceTool


## How to use ?

Please change the direction of the results you want to analyze 

In [2]:
# to change
path = 'severalexp1'
number = 1
# end of to change

In [3]:
results_dir = os.path.join(local_dir, 'results', '{}'.format(path), 'id_{}'.format(number))
haploid_dir = os.path.join(results_dir, 'haploid')
diploid_dir = os.path.join(results_dir, 'diploid')

file = 'configs.json'
haploid_json_dir_file = os.path.join(haploid_dir, 'json', file)
diploid_json_dir_file = os.path.join(diploid_dir, 'json', file)

with open(haploid_json_dir_file, 'r') as f :
    haploid_data_config = json.load(f)

with open(diploid_json_dir_file, 'r') as f :
    diploid_data_config = json.load(f)

config = Config(haploid_data_config)
distance_tool = DistanceTool(config)

result = ''
for key, value in haploid_data_config.items():
    result += f'{key} : {value}\n'

diploid_result = ''
for key, value in diploid_data_config.items():
    diploid_result += f'{key} : {value}\n'

print(result)
print(diploid_result)

number_of_exp : 3
number_of_config : 5
threshold_var : 0.15
body_shape : [5, 5]
max_weight : 1
max_bias : 1
response : 1
max_weight_cppn : 20
max_bias_cppn : 20
range_weight : 1
range_bias : 1
sigma_weight : 0.2
sigma_bias : 0.3
threshold_weight : 0.3
threshold_bias : 0.15
threshold_function : 0.2
threshold_dominance : 0.2
number_of_dominances : 5
generations : 150
population : 128
number_of_winner : 1
number_in_tournament : 10
number_of_elites : 10
shape_of_cppn : [[7], [5], [3, 3], [1, 1]]
function_pool : ['gaussian', 'sin', 'tanh']
n_steps : 1200
number_of_reported_individuals : 40
cpus : 20
env_name : CaveCrawler-v0

number_of_exp : 3
number_of_config : 5
threshold_var : 0.15
body_shape : [5, 5]
max_weight : 1
max_bias : 1
response : 1
max_weight_cppn : 20
max_bias_cppn : 20
range_weight : 1
range_bias : 1
sigma_weight : 0.2
sigma_bias : 0.3
threshold_weight : 0.3
threshold_bias : 0.15
threshold_function : 0.2
threshold_dominance : 0.2
number_of_dominances : 5
generations : 150
pop

## Study with graph


In [ ]:
general_normalized_activation_function_distance_haploid = []
general_normalized_weight_distance_haploid = []
general_normalized_bias_distance_haploid = []
general_normalized_distance_bodies_haploid = []
general_normalized_activation_function_distance_diploid = []
general_normalized_weight_distance_diploid = []
general_normalized_bias_distance_diploid = []
general_normalized_distance_bodies_diploid = []


for k in range(haploid_data_config['number_of_config']) :
    general_normalized_activation_function_distance_haploid_sigma = []
    general_normalized_weight_distance_haploid_sigma = []
    general_normalized_bias_distance_haploid_sigma = []
    general_normalized_distance_bodies_haploid_sigma = []
    general_normalized_activation_function_distance_diploid_sigma = []
    general_normalized_weight_distance_diploid_sigma = []
    general_normalized_bias_distance_diploid_sigma = []
    general_normalized_distance_bodies_diploid_sigma = []
    for i in range(diploid_data_config['number_of_exp']) :
        id_exp = number + k * diploid_data_config['number_of_exp'] + i
        results_dir = os.path.join(local_dir, 'results', '{}'.format(path), 'id_{}'.format(id_exp))
        haploid_dir = os.path.join(results_dir, 'haploid')
        diploid_dir = os.path.join(results_dir, 'diploid')

        generation = 1
        pkl_dir = os.path.join(haploid_dir, 'pkl')
        generations = []
        average_list_normalized_activation_function_distance = []
        average_list_normalized_weight_distance = []
        average_list_normalized_bias_distance = []
        average_list_normalized_distance_bodies = []

        while True :

            cppn = [f for f in os.listdir(pkl_dir) if f.startswith('{}_cppn_registry.pkl'.format(generation))]
            body = [f for f in os.listdir(pkl_dir) if f.startswith('{}_body_registry.pkl'.format(generation))]

            if not cppn or not body :
                break

            file_dir = os.path.join(pkl_dir, cppn[0])
            with open(file_dir, 'rb') as f :
                loaded_file_cppn = dill.load(f)

            file_dir = os.path.join(pkl_dir, body[0])
            with open(file_dir, 'rb') as f :
                loaded_file_body = dill.load(f)

            list_of_keys_cppn = list(loaded_file_cppn.keys())
            list_of_keys_body = list(loaded_file_body.keys())

            normalized_act_function_distances = []
            normalized_weight_distances = []
            normalized_bias_distances = []
            normalized_distance_bodys = []

            for i in range(len(list_of_keys_cppn)) :
                for j in range(i+1, len(list_of_keys_cppn)) :
                    generation1, id_cppn1 = list_of_keys_cppn[i]
                    generation2, id_cppn2 = list_of_keys_cppn[j]

                    generation1_body, id_body1 = list_of_keys_body[i]
                    generation2_body, id_body2 = list_of_keys_body[j]

                    node_evals1 = loaded_file_cppn[(generation1, id_cppn1)].node_evals
                    node_evals2 = loaded_file_cppn[(generation2, id_cppn2)].node_evals
                    body1 = loaded_file_body[(generation1_body, id_body1)].body
                    body2 = loaded_file_body[(generation2_body, id_body2)].body

                    act_function_distance, weight_distance, bias_distance, normalized_act_function_distance, normalized_weight_distance, normalized_bias_distance = distance_tool.distance_expressed_genome(node_evals1, node_evals2)
                    distance, distance_normalized = distance_tool.phenotypic_body_distance(body1, body2)

                    normalized_act_function_distances.append(normalized_act_function_distance)
                    normalized_weight_distances.append(normalized_weight_distance)
                    normalized_bias_distances.append(normalized_bias_distance)
                    normalized_distance_bodys.append(distance_normalized)

            average_normalized_act_function_distance = np.mean(np.array(normalized_act_function_distances))
            average_normalized_weight_distance = np.mean(np.array(normalized_weight_distances))
            average_normalized_bias_distance = np.mean(np.array(normalized_bias_distances))
            average_normalized_distance_body = np.mean(np.array(normalized_distance_bodys))

            average_list_normalized_activation_function_distance.append(average_normalized_act_function_distance)
            average_list_normalized_weight_distance.append(average_normalized_weight_distance)
            average_list_normalized_bias_distance.append(average_normalized_bias_distance)
            average_list_normalized_distance_bodies.append(average_normalized_distance_body)

            generations.append(generation)
            generation += 1

        generation = 1
        pkl_dir = os.path.join(diploid_dir, 'pkl')
        diploid_generations = []
        diploid_average_list_normalized_activation_function_distance = []
        diploid_average_list_normalized_weight_distance = []
        diploid_average_list_normalized_bias_distance = []
        diploid_average_list_normalized_distance_bodies = []

        while True :

            cppn = [f for f in os.listdir(pkl_dir) if f.startswith('{}_cppn_registry.pkl'.format(generation))]
            body = [f for f in os.listdir(pkl_dir) if f.startswith('{}_body_registry.pkl'.format(generation))]

            if not cppn or not body :
                break

            file_dir = os.path.join(pkl_dir, cppn[0])
            with open(file_dir, 'rb') as f :
                loaded_file_cppn = dill.load(f)

            file_dir = os.path.join(pkl_dir, body[0])
            with open(file_dir, 'rb') as f :
                loaded_file_body = dill.load(f)

            list_of_keys_cppn = list(loaded_file_cppn.keys())
            list_of_keys_body = list(loaded_file_body.keys())

            normalized_act_function_distances = []
            normalized_weight_distances = []
            normalized_bias_distances = []
            normalized_distance_bodys = []

            for i in range(len(list_of_keys_cppn)) :
                for j in range(i+1, len(list_of_keys_cppn)) :
                    generation1, id_cppn1 = list_of_keys_cppn[i]
                    generation2, id_cppn2 = list_of_keys_cppn[j]

                    generation1_body, id_body1 = list_of_keys_body[i]
                    generation2_body, id_body2 = list_of_keys_body[j]

                    node_evals1 = loaded_file_cppn[(generation1, id_cppn1)].node_evals
                    node_evals2 = loaded_file_cppn[(generation2, id_cppn2)].node_evals
                    body1 = loaded_file_body[(generation1_body, id_body1)].body
                    body2 = loaded_file_body[(generation2_body, id_body2)].body

                    act_function_distance, weight_distance, bias_distance, normalized_act_function_distance, normalized_weight_distance, normalized_bias_distance = distance_tool.distance_expressed_genome(node_evals1, node_evals2)
                    distance, distance_normalized = distance_tool.phenotypic_body_distance(body1, body2)

                    normalized_act_function_distances.append(normalized_act_function_distance)
                    normalized_weight_distances.append(normalized_weight_distance)
                    normalized_bias_distances.append(normalized_bias_distance)
                    normalized_distance_bodys.append(distance_normalized)

            average_normalized_act_function_distance = np.mean(np.array(normalized_act_function_distances))
            average_normalized_weight_distance = np.mean(np.array(normalized_weight_distances))
            average_normalized_bias_distance = np.mean(np.array(normalized_bias_distances))
            average_normalized_distance_body = np.mean(np.array(normalized_distance_bodys))

            diploid_average_list_normalized_activation_function_distance.append(average_normalized_act_function_distance)
            diploid_average_list_normalized_weight_distance.append(average_normalized_weight_distance)
            diploid_average_list_normalized_bias_distance.append(average_normalized_bias_distance)
            diploid_average_list_normalized_distance_bodies.append(average_normalized_distance_body)

            diploid_generations.append(generation)
            generation += 1

        threshold = haploid_data_config['threshold_weight'] + k * haploid_data_config['threshold_var']
        plt.figure()
        plt.title('threshold {:.2f}'.format(threshold))
        plt.plot(generations, average_list_normalized_activation_function_distance, label = 'Average normalized distance between the activation functions of the cppns, haploid')
        plt.plot(generations, average_list_normalized_weight_distance, label = 'Average normalized distance between the weights of the cppns, haploid')
        plt.plot(generations, average_list_normalized_bias_distance, label = 'Average normalized distance between the biases of the cppns, haploid')
        plt.plot(generations, average_list_normalized_distance_bodies, label = 'Average normalized distance between the bodies, haploid')
        plt.plot(diploid_generations, diploid_average_list_normalized_activation_function_distance, label = 'Average normalized distance between the activation functions of the cppns, diploid')
        plt.plot(diploid_generations, diploid_average_list_normalized_weight_distance, label = 'Average normalized distance between the weights of the cppns, diploid')
        plt.plot(diploid_generations, diploid_average_list_normalized_bias_distance, label = 'Average normalized distance between the biases of the cppns, diploid')
        plt.plot(diploid_generations, diploid_average_list_normalized_distance_bodies, label = 'Average normalized distance between the bodies, diploid')
        plt.xlabel('Generations')
        plt.ylabel('Normalized distance')
        plt.grid(True, which = 'both')
        plt.legend(bbox_to_anchor=(1,1), loc = 'upper left')
        plt.show()

        general_normalized_activation_function_distance_haploid_sigma.append(average_list_normalized_activation_function_distance)
        general_normalized_weight_distance_haploid_sigma.append(average_list_normalized_weight_distance)
        general_normalized_bias_distance_haploid_sigma.append(average_list_normalized_bias_distance)
        general_normalized_distance_bodies_haploid_sigma.append(average_list_normalized_distance_bodies)
        general_normalized_activation_function_distance_diploid_sigma.append(diploid_average_list_normalized_activation_function_distance)
        general_normalized_weight_distance_diploid_sigma.append(diploid_average_list_normalized_weight_distance)
        general_normalized_bias_distance_diploid_sigma.append(diploid_average_list_normalized_bias_distance)
        general_normalized_distance_bodies_diploid_sigma.append(diploid_average_list_normalized_distance_bodies)

    general_normalized_activation_function_distance_haploid.append(general_normalized_activation_function_distance_haploid_sigma)
    general_normalized_weight_distance_haploid.append(general_normalized_weight_distance_haploid_sigma)
    general_normalized_bias_distance_haploid.append(general_normalized_bias_distance_haploid_sigma)
    general_normalized_distance_bodies_haploid.append(general_normalized_distance_bodies_haploid_sigma)
    general_normalized_activation_function_distance_diploid.append(general_normalized_activation_function_distance_diploid_sigma)
    general_normalized_weight_distance_diploid.append(general_normalized_weight_distance_diploid_sigma)
    general_normalized_bias_distance_diploid.append(general_normalized_bias_distance_diploid_sigma)
    general_normalized_distance_bodies_diploid.append(general_normalized_distance_bodies_diploid_sigma)

In [ ]:
for k in range(haploid_data_config['number_of_config']) :
    threshold = haploid_data_config['threshold_weight'] + k * haploid_data_config['threshold_var']

    mean_normalized_activation_function_distance_haploid = np.mean(np.array(general_normalized_activation_function_distance_haploid[k]), axis = 0)
    mean_normalized_weight_distance_haploid = np.mean(np.array(general_normalized_weight_distance_haploid[k]), axis = 0)
    mean_normalized_bias_distance_haploid = np.mean(np.array(general_normalized_bias_distance_haploid[k]), axis = 0)
    mean_normalized_distance_bodies_haploid = np.mean(np.array(general_normalized_distance_bodies_haploid[k]), axis = 0)
    mean_normalized_activation_function_distance_diploid = np.mean(np.array(general_normalized_activation_function_distance_diploid[k]), axis = 0)
    mean_normalized_weight_distance_diploid = np.mean(np.array(general_normalized_weight_distance_diploid[k]), axis = 0)
    mean_normalized_bias_distance_diploid = np.mean(np.array(general_normalized_bias_distance_diploid[k]), axis = 0)
    mean_normalized_distance_bodies_diploid = np.mean(np.array(general_normalized_distance_bodies_diploid[k]), axis = 0)

    std_normalized_activation_function_distance_haploid = np.std(np.array(general_normalized_activation_function_distance_haploid[k]), axis = 0)
    std_normalized_weight_distance_haploid = np.std(np.array(general_normalized_weight_distance_haploid[k]), axis = 0)
    std_normalized_bias_distance_haploid = np.std(np.array(general_normalized_bias_distance_haploid[k]), axis = 0)
    std_normalized_distance_bodies_haploid = np.std(np.array(general_normalized_distance_bodies_haploid[k]), axis = 0)
    std_normalized_activation_function_distance_diploid = np.std(np.array(general_normalized_activation_function_distance_diploid[k]), axis = 0)
    std_normalized_weight_distance_diploid = np.std(np.array(general_normalized_weight_distance_diploid[k]), axis = 0)
    std_normalized_bias_distance_diploid = np.std(np.array(general_normalized_bias_distance_diploid[k]), axis = 0)
    std_normalized_distance_bodies_diploid = np.std(np.array(general_normalized_distance_bodies_diploid[k]), axis = 0)

    plt.figure()
    plt.title('threshold {:.2f}'.format(threshold))
    plt.plot(generations, mean_normalized_activation_function_distance_haploid, label = 'Average normalized distance between the activation functions of the cppns, haploid')
    plt.fill_between(generations, mean_normalized_activation_function_distance_haploid - std_normalized_activation_function_distance_haploid, mean_normalized_activation_function_distance_haploid + std_normalized_activation_function_distance_haploid, alpha = 0.3)
    plt.plot(generations, mean_normalized_weight_distance_haploid, label = 'Average normalized distance between the weights of the cppns, haploid')
    plt.fill_between(generations, mean_normalized_weight_distance_haploid - std_normalized_weight_distance_haploid, mean_normalized_weight_distance_haploid + std_normalized_weight_distance_haploid, alpha = 0.3)
    plt.plot(generations, mean_normalized_bias_distance_haploid, label = 'Average normalized distance between the biases of the cppns, haploid')
    plt.fill_between(generations, mean_normalized_bias_distance_haploid - std_normalized_bias_distance_haploid, mean_normalized_bias_distance_haploid + std_normalized_bias_distance_haploid, alpha = 0.3)
    plt.plot(generations, mean_normalized_distance_bodies_haploid, label = 'Average normalized distance between the bodies, haploid')
    plt.fill_between(generations, mean_normalized_distance_bodies_haploid - std_normalized_distance_bodies_haploid, mean_normalized_distance_bodies_haploid + std_normalized_distance_bodies_haploid, alpha = 0.3)
    plt.plot(diploid_generations, mean_normalized_activation_function_distance_diploid, label = 'Average normalized distance between the activation functions of the cppns, diploid')
    plt.fill_between(diploid_generations, mean_normalized_activation_function_distance_diploid - std_normalized_activation_function_distance_diploid, mean_normalized_activation_function_distance_diploid + std_normalized_activation_function_distance_diploid, alpha = 0.3)
    plt.plot(diploid_generations, mean_normalized_weight_distance_diploid, label = 'Average normalized distance between the weights of the cppns, diploid')
    plt.fill_between(diploid_generations, mean_normalized_weight_distance_diploid - std_normalized_weight_distance_diploid, mean_normalized_weight_distance_diploid + std_normalized_weight_distance_diploid, alpha = 0.3)
    plt.plot(diploid_generations, mean_normalized_bias_distance_diploid, label = 'Average normalized distance between the biases of the cppns, diploid')
    plt.fill_between(diploid_generations, mean_normalized_bias_distance_diploid - std_normalized_bias_distance_diploid, mean_normalized_bias_distance_diploid + std_normalized_bias_distance_diploid, alpha = 0.3)
    plt.plot(diploid_generations, mean_normalized_distance_bodies_diploid, label = 'Average normalized distance between the bodies, diploid')
    plt.fill_between(diploid_generations, mean_normalized_distance_bodies_diploid - std_normalized_distance_bodies_diploid, mean_normalized_distance_bodies_diploid + std_normalized_distance_bodies_diploid, alpha = 0.3)
    plt.xlabel('Generations')
    plt.ylabel('Normalized distance')
    plt.grid(True, which = 'both')
    plt.legend(bbox_to_anchor=(1,1), loc = 'upper left')
    plt.show()